# dev26 미래·라벨 불변성과기준선갱신 점검


In [1]:
import nbformat,json
from pathlib import Path
import numpy as np,pandas as pd
root=Path.cwd()
n=nbformat.read(root/"notebooks/181_cn7_dev26_supervised_gmm.ipynb",as_version=4)
s="\n".join(c.source for c in n.cells if c.cell_type=="code")
ns={}
exec(s[:s.index("MODELS=")],ns)
a=ns["d"].iloc[:90].copy().reset_index(drop=True)
segments=np.zeros(len(a),int)
before=ns["dev26"](a,segments)
changed=a.copy()
for c in ns["RAW"]:changed.loc[60:,c+"_raw"]*=2
after=ns["dev26"](changed,segments)
assert np.allclose(before[:60],after[:60],equal_nan=True)
label_changed=a.copy();label_changed["gas"]=~label_changed.gas
z=ns["dev26"](label_changed,segments)
assert np.allclose(before,z,equal_nan=True)
x=np.r_[np.zeros(5),np.full(5,10.),10.]
r=ns["causal_robust_dev"](x,1.)
assert np.isnan(r[:5]).all() and (r[5:10]==10).all() and r[10]==0
u=pd.read_csv(root/"reports/tables/cn7_179_unlabeled_predictions.csv")
unknown=~u.known_label
proof={"future_changes_do_not_alter_prefix":True,"label_changes_do_not_alter_features":True,
 "feature_count":before.shape[1],"reject5_then_reset_check":True,
 "unlabeled_unknown":int(unknown.sum()),
 "unknown_max_screw_rpm_nonpositive":int((unknown&u.Max_Screw_RPM_raw.le(0)).sum()),
 "common_py_reproduction":"not exact; user definition plus existing causal_robust_dev; fixed recording units"}
for file in ["cn7_181_dev26_gmm_training_audit.csv","cn7_182_dev26_gmm_training_audit.csv"]:
 p=root/"reports/tables"/file
 if p.exists():
  q=pd.read_csv(p)
  assert q.timestamp_overlap.eq(0).all() and q.test_raw_in_training_history.eq(0).all()
  proof[file]={"fits":len(q),"timestamp_overlap":0,"test_raw_in_training_history":0}
(root/"reports/tables/cn7_dev26_causal_feature_checks.json").write_text(json.dumps(proof,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(proof,ensure_ascii=False,indent=2))


{
  "future_changes_do_not_alter_prefix": true,
  "label_changes_do_not_alter_features": true,
  "feature_count": 26,
  "reject5_then_reset_check": true,
  "unlabeled_unknown": 28138,
  "unknown_max_screw_rpm_nonpositive": 18124,
  "common_py_reproduction": "not exact; user definition plus existing causal_robust_dev; fixed recording units",
  "cn7_181_dev26_gmm_training_audit.csv": {
    "fits": 220,
    "timestamp_overlap": 0,
    "test_raw_in_training_history": 0
  }
}
